In [9]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
import asyncio

In [2]:
load_dotenv(override=True)
model = "gpt-4o-mini"

In [3]:
instruction_professional = """
You are a sales agent working for ComplAI, a company that provide SaaS tool for ensurinf SOC2 complaince
and preparing for audits, powered by AI.
You write professional, serious cold emails.
"""

instruction_engaging = """
You are a sales agent working for ComplAI, a company that provide SaaS tool for ensurinf SOC2 complaince
and preparing for audits, powered by AI.
You write witty, engaging cold emails that are likely to get a response.
"""

instruction_busy = """
You are a sales agent working for ComplAI, a company that provide SaaS tool for ensurinf SOC2 complaince
and preparing for audits, powered by AI.
You write concise, to the point cold emails.
"""

In [4]:
sales_agent_profession = Agent(
    name="Professional Sales Agent",
    instructions=instruction_professional,
    model=model
)
sales_agent_engaging = Agent(
    name="Enganing Sales Agent",
    instructions=instruction_engaging,
    model=model
)
sales_agent_busy = Agent(
    name="Busy Sales Agent",
    instructions=instruction_busy,
    model=model
)

In [12]:
result = Runner.run_streamed(starting_agent=sales_agent_profession, input="Write a cold sales email.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with ComplAI

Hi [Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a cutting-edge SaaS tool designed to streamline SOC 2 compliance and audit preparation.

As you know, maintaining compliance can be a complex and time-consuming process, often diverting valuable resources from your core business objectives. Our AI-powered platform simplifies this challenge by automating key tasks, ensuring that your documentation, controls, and processes are up to date and aligned with the latest standards.

Here are a few ways ComplAI can benefit your organization:

- **Automated Documentation**: Our tool generates the necessary compliance documentation, saving your team hours of manual work.
- **Real-Time Monitoring**: With continuous oversight of your controls, you can identify and address potential compliance issues before they escalate.
- **Audit Preparation**: Simplify the audit process with our organized, easy-to-

In [7]:
message = "Write a cold sales email."

with trace("Parallel Cold Emails"):
    results = await asyncio.gather(
        Runner.run(starting_agent=sales_agent_profession, input=message),
        Runner.run(starting_agent=sales_agent_engaging, input=message),
        Runner.run(starting_agent=sales_agent_busy, input=message)
    )

outputs = [output.final_output for output in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Hi [Recipient’s Name],

I hope this message finds you well.

I’m reaching out to introduce you to ComplAI, a specialized SaaS solution designed to simplify the SOC 2 compliance process and optimize your audit preparations. As organizations increasingly rely on data security measures, ensuring compliance is more critical than ever.

ComplAI leverages cutting-edge AI technology to automate documentation, track compliance statuses, and prepare your team for audits, ultimately saving you time and reducing the risk of non-compliance. Our platform provides:

- Real-time monitoring and alerts for compliance issues.
- Comprehensive documentation management.
- Seamless integration with your existing systems.

Many companies like yours have found great success in using ComplAI, experiencing a significant reduction in the time and resources spent on compliance activities.

I’d love to schedule a brief call to discuss how ComplAI can support 

In [5]:
instruction_sales_picker = """
You pick the best cold email from the given options.
Imagine you are a customer and pick the one you are most likey to respond to.
Do not give an explanation; only reply with the selected email.
"""
sales_agent_picker = Agent(
    name="Sales Picker Agent",
    instructions=instruction_sales_picker,
    model=model
)

In [8]:
message = "Write a cold sales email."

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(starting_agent=sales_agent_profession, input=message),
        Runner.run(starting_agent=sales_agent_engaging, input=message),
        Runner.run(starting_agent=sales_agent_busy, input=message)
    )

outputs = [output.final_output for output in results]

cold_emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
best_email = await Runner.run(starting_agent=sales_agent_picker, input=cold_emails)

print(f"Best sales email:\n\n{best_email.final_output}")

Best sales email:

Subject: 🚀 Is Your SOC 2 Compliance Ready for Liftoff? 

Hi [First Name],

I hope this email finds you swimming in a sea of productivity, but I also know how easily compliance can feel like a tidal wave crashing down—especially when it’s SOC 2 season!

What if I told you that handling SOC 2 compliance could be as easy as a quick stroll in your favorite park? At ComplAI, we’ve designed an AI-powered tool that transforms the audit prep chaos into a seamless experience.

Our secret sauce? We handle the heavy lifting for you, so your team can focus on what they do best—like perfecting that coffee blend or brainstorming the next big feature!

And don’t worry; we promise there’s no fine print in the fine print. Just clarity, insights, and maybe a few laugh lines along the way.

Want to see how we can turn audit nightmares into dreams? Let’s chat! 

Best,  
[Your Name]  
[Your Title]  
ComplAI  
[Your Contact Information]  

P.S. I hear compliance is more fun with a dash of

In [12]:
@function_tool
def write_file(email: str):
    """ Write the email to a file """
    file = "email.txt"
    with open(file=file, mode="w") as f:
        f.write(email)

In [13]:
write_file

FunctionTool(name='write_file', description='Write the email to a file', params_json_schema={'properties': {'email': {'title': 'Email', 'type': 'string'}}, 'required': ['email'], 'title': 'write_file_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7e444bfb3100>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [14]:
tool = sales_agent_profession.as_tool(tool_name="sales_agent_profession", tool_description="Write a professional cold sales email")

In [15]:
tool

FunctionTool(name='sales_agent_profession', description='Write a professional cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent_profession_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7e444951efc0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [20]:
tools = [
    sales_agent_profession.as_tool(tool_name="sales_agent_profession", tool_description="Write a professional cold sales email"),
    sales_agent_engaging.as_tool(tool_name="sales_agent_engaging", tool_description="Write a engaging cold sales email"),
    sales_agent_busy.as_tool(tool_name="sales_agent_busy", tool_description="Write a busy cold sales email"),
    write_file
]

In [21]:
tools

[FunctionTool(name='sales_agent_profession', description='Write a professional cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent_profession_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7e4448bb4360>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent_engaging', description='Write a engaging cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent_engaging_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x7e4448bb5080>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrai

In [23]:
instructions_sales_manager = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the write_file tool to write the best email (and only the best email) to the file.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must write ONE email using the write_file tool — never more than one.
"""

sales_manager_agent = Agent(
    name="Sales Manager Agent",
    instructions=instructions_sales_manager,
    model=model,
    tools=tools
)

message = "Draft a cold sales email to 'Dear CEO'."
with trace("Sales Manager"):
    result = await Runner.run(starting_agent=sales_manager_agent, input=message)